In [ ]:
# --- Set up Kaggle API credentials using Colab Secrets ----
import os
from google.colab import userdata

try:
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Kaggle credentials loaded from Colab Secrets.")
except userdata.SecretValueError:
    print("WARNING: KAGGLE_USERNAME or KAGGLE_KEY not found in Colab Secrets.")
    print("Please add them using the 'Secrets' tab on the left sidebar.")


In [ ]:
import os
import zipfile

# Install Kaggle API client if not already installed
!pip install kaggle --upgrade

# The Kaggle CLI will automatically pick up KAGGLE_USERNAME and KAGGLE_KEY from environment variables
# which are set by the 'Colab Secrets' cell.

# Define the target directory for the dataset
dataset_dir = 'datasets/predicting-car-prices-from-text'
os.makedirs(dataset_dir, exist_ok=True)

# Download the competition dataset
# The competition name is 'predicting-car-prices-from-text-11-5-25'
# The '-p' flag specifies the path where files will be downloaded.
print(f"Downloading competition data to {dataset_dir}...")
!kaggle competitions download -c predicting-car-prices-from-text-11-5-25 -p {dataset_dir} --force # Use --force to redownload if needed

# Unzip the downloaded files
# Kaggle usually downloads a zip file with the competition name.
# We need to find the zip file and unzip it.
downloaded_files = os.listdir(dataset_dir)
zip_file_name = None
for f in downloaded_files:
    if f.endswith('.zip'):
        zip_file_name = f
        break

if zip_file_name:
    zip_file_path = os.path.join(dataset_dir, zip_file_name)
    print(f"Extracting {zip_file_name}...")
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)
    print(f"Extracted contents to {dataset_dir}")
    # Optionally, remove the zip file after extraction
    !rm {zip_file_path}
else:
    print("No zip file found in the dataset directory to extract.")

# Verify if train_text.csv exists
if os.path.exists(os.path.join(dataset_dir, 'train_text.csv')):
    print(f"train_text.csv found at {os.path.join(dataset_dir, 'train_text.csv')}")
    print("Dataset download and extraction completed successfully!")
else:
    print("Error: train_text.csv was not found after extraction. Please check the Kaggle competition files.")


 79% 2.00M/2.55M [00:01<00:00, 2.14MB/s]
100% 2.55M/2.55M [00:01<00:00, 2.27MB/s]
Extracting predicting-car-prices-from-text-11-5-25.zip...
Extracted contents to datasets/predicting-car-prices-from-text
train_text.csv found at datasets/predicting-car-prices-from-text/train_text.csv
Dataset download and extraction completed successfully!


In [ ]:
# Let's load the data set that we will be working with.
import pandas as pd

# Load dataset
df = pd.read_csv('datasets/predicting-car-prices-from-text/train_text.csv')

print(f"Dataset size: {len(df)} cars")
print(f"Columns: {list(df.columns)}")
print(f"Price range: ${df['price'].min()} - ${df['price'].max()}")
print(
    f"Description length: {df['description'].str.len().min()} - {df['description'].str.len().max()}"
)
print(f"Mean price: ${df['price'].mean():.2f}")
print(f"Median price: ${df['price'].median():.2f}")

for i in range(3):
    print(f"Example {i+1}:")
    print(f"Description: {df.iloc[i]['description'][:200]}...")
    print(f"Price: ${df.iloc[i]['price']:.2f}")

Dataset size: 27136 cars
Columns: ['id', 'description', 'price']
Price range: $2000 - $2954083
Description length: 215 - 807
Mean price: $39396.98
Median price: $28000.00
Example 1:
Description: Title status: Yes. Running on Gasoline with a 518.0HP Electric Motor Electric Fuel System, this 2022 Tesla Model X Long Range is available in Chicago. Features A/T transmission and has 7,500 miles. In...
Price: $64000.00
Example 2:
Description: Clean title: Yes. Gray exterior with Black interior. Porsche's 2024 Cayenne GTS Coupe AWD currently in New York. Features: 8-Speed A/T transmission, Gasoline fuel system, and 453.0HP 4.0L 8 Cylinder E...
Price: $145000.00
Example 3:
Description: Available in ATLANTA: This meticulously maintained 2006 Chevrolet Silverado 1500 LT Crew Cab with title status Yes. The exterior is finished in stunning Black, perfectly complemented by a luxurious Bl...
Price: $6995.00


In [ ]:
# Let's convert text to the word-level tokens.
from collections import Counter
import torch


class WordTokenizer:

    def __init__(self, vocab_size=512):
        self.vocab_size = vocab_size
        self.vocab = {'<PAD>': 0, '<UNK>': 1}
        self.max_len = 128

    def build_from_text(self, text):
        word_counts = Counter()
        for line in text:
            words = line.lower().replace(',', ' ').replace('.', ' ').split()
            word_counts.update(words)
        for word, _ in word_counts.most_common(self.vocab_size - 2):
            self.vocab[word] = len(self.vocab)

    def encode(self, text):
        tokens = []
        words = text.lower().replace(',', ' ').replace('.', ' ').split()
        for word in words[:self.max_len]:
            tokens.append(self.vocab.get(word, self.vocab['<UNK>']))
        if len(tokens) < self.max_len:
            tokens += [self.vocab['<PAD>']] * (self.max_len - len(tokens))
        return torch.tensor(tokens)

    def decode(self, token_ids):
        inv_vocab = {v: k for k, v in self.vocab.items()}
        words = [inv_vocab.get(token_id, '<UNK>') for token_id in token_ids]
        return ' '.join(words).replace(' <PAD>', '')

    def info(self):
        print(f"Vocabulary size: {len(self.vocab)} words")
        for word in ['1999', '2022', 'bmw', 'sedan', 'awd']:
            token_id = self.vocab.get(word, self.vocab['<UNK>'])
            status = 'ok' if word in self.vocab else 'unknown'
            print(f"  '{word}': {token_id} {status}")


word_tokenizer = WordTokenizer(vocab_size=1500)
word_tokenizer.build_from_text(df['description'])
word_tokenizer.info()

text = "2018 BMW x7"
tokens = word_tokenizer.encode(text)

print(f"Text: {text}")
print(f"Tokens shape: {tokens.shape}")
print(f"Tokens: {tokens[:10]}")
print(f"Decoded: {word_tokenizer.decode(tokens.tolist())}")

Vocabulary size: 1500 words
  '1999': 763 ok
  '2022': 133 ok
  'bmw': 100 ok
  'sedan': 1 unknown
  'awd': 918 ok
Text: 2018 BMW x7
Tokens shape: torch.Size([128])
Tokens: tensor([116, 100, 697,   0,   0,   0,   0,   0,   0,   0])
Decoded: 2018 bmw x7


In [ ]:
# Let's create a number tokinizer, like a P10.
import numpy as np


class NumberTokenizer:

    def __init__(self, num_digits=4):
        self.num_digits = num_digits
        self.tokens = [
            '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '.', 'E', '+',
            '-', '<PAD>', '<BOS>'
        ]
        self.vocab = {token: idx for idx, token in enumerate(self.tokens)}
        self.inv_vocab = {idx: token for token, idx in self.vocab.items()}

    def encode(self, number):
        num_str = np.format_float_scientific(
            number,
            precision=self.num_digits - 1,
            min_digits=self.num_digits - 1,
            sign=True  # Always include sign
        )
        tokens = [self.vocab['<BOS>']]
        for ch in num_str:
            if ch in self.vocab:
                tokens.append(self.vocab[ch])
            elif ch == 'e':
                tokens.append(self.vocab['E'])
        return torch.tensor(tokens)

    def possible_next_tokens(self, num_prev_tokens):
        """Returns list of valid token IDs for the next position.

        Format: +d.dddE+dd (exactly 10 chars for num_digits=4)
        Positions after BOS: 0=sign, 1=digit, 2='.', 3-5=digits, 6='E', 7=sign, 8-9=2 exp digits
        """
        index = num_prev_tokens
        num_exp_digits = 2

        if index == 0:  # First token after BOS: must be sign
            return [self.vocab['+'], self.vocab['-']]
        elif index == 1:  # First digit before decimal point
            return [self.vocab[str(i)] for i in range(10)]
        elif index == 2:  # Decimal point
            return [self.vocab['.']]
        elif index < 2 + self.num_digits:  # Mantissa digits after decimal (positions 3-5)
            return [self.vocab[str(i)] for i in range(10)]
        elif index == 2 + self.num_digits:  # Exponent marker 'E'
            return [self.vocab['E']]
        elif index == 3 + self.num_digits:  # Exponent sign
            return [self.vocab['+'], self.vocab['-']]
        elif index < 4 + self.num_digits + num_exp_digits:  # Exactly 2 exponent digits (positions 8-9)
            return [self.vocab[str(i)] for i in range(10)]
        else:
            return [self.vocab['<PAD>']]

    def decode(self, tokens):
        num_str = ''
        for token in tokens[1:]:
            token_val = token.item() if torch.is_tensor(token) else token
            if token_val == self.vocab['<PAD>']:
                break
            char = self.inv_vocab[token_val]
            if char not in ['<BOS>', '<PAD>']:
                num_str += 'e' if char == 'E' else char

        try:
            return float(num_str)
        except ValueError:
            return 0.0


num_tokenizer = NumberTokenizer(num_digits=4)

test_val = 27185
tokens = num_tokenizer.encode(test_val) # +2.718E+04
decoded = num_tokenizer.decode(tokens)
print(f"  Input:   {test_val}")
print(f"  Tokens:  {tokens.tolist()}")
print(f"  Decoded: {decoded:.6f}")

  Input:   27185
  Tokens:  [15, 12, 2, 10, 7, 1, 8, 11, 12, 0, 4]
  Decoded: 27180.000000


In [ ]:
# Simplified Transformer encoder

from torch import nn


class TransformerEncoder(nn.Module):

    def __init__(self,
                 vocab_size=256,
                 d_model=128,
                 n_head=4,
                 num_layers=2,
                 max_len=128):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(
            vocab_size,
            d_model)  # shape = (1500, 128), 1500 × 128 = 192,000 parameters
        pe = torch.zeros(max_len, d_model)  # shape = (128, 128)
        position = torch.arange(0, max_len).unsqueeze(1)  # shape = (128, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-np.log(10000.0) / d_model))  # shape = (64,)
        pe[:, 0::2] = torch.sin(position * div_term)  # shape = (128, 64) for even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # shape = (128, 64) for odd indices
        self.register_buffer('pe', pe)
        # 1. We have 4 (n_head) attention heads:
        #    - Q K, V and O: with 128 * 128 + 128 (d_model * d_model + d_model)
        #      shape which gives us 4 * (128 * 128 + 128) = 66,048 parameters per layer.
        # 2. We have 2 LayerNorm layers per encoder layer with 128 + 128 (d_model + d_model)
        # shape which gives us 2 * (128 + 128) = 512 parameters per layer.
        # 3. We have a feed-forward network with two linear layers (d_model -> 4*d_model -> d_model):
        #    - First layer: (128 × 512 + 512) = 66,560 parameters
        #    - Second layer: (512 × 128 + 128) = 65,664 parameters
        #    - Total: 66,560 + 65,664 = 131,712 parameters per layer.
        # Total per layer: 66,048 + 512 + 131,712 = 198,272 parameters per layer.
        # For 2 layers: 2 * 198,272 = 396,544 parameters for all encoder layers.
        # Final LayerNorm: 128 + 128 = 256 parameters.
        # Total parameters: 192,000 + 396,544 + 256 = 588,800 parameters.
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model,
                                                   nhead=n_head,
                                                   dim_feedforward=4 * d_model,
                                                   dropout=0.1,
                                                   batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer,
                                                 num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, src_key_padding_mask=None):
        seq_len = x.shape[1]
        x = self.embedding(x) * np.sqrt(self.d_model)
        x = x + self.pe[:seq_len, :].unsqueeze(0)
        x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)
        return self.norm(x)


# Test the improved encoder
encoder = TransformerEncoder(vocab_size=len(word_tokenizer.vocab), d_model=128)

sample_text = "2018 BMW x7"
tokens = word_tokenizer.encode(sample_text)

# Create padding mask (True for padding tokens)
padding_mask = (tokens == word_tokenizer.vocab['<PAD>'])

print(f"Input text: '{sample_text}'")
print(f"Text tokens shape: {tokens.shape}")
print(f"Padding mask shape: {padding_mask.shape}")
print(f"Number of non-padding tokens: {(~padding_mask).sum().item()}")

# Encode with batch dimension and padding mask
encoded = encoder(tokens.unsqueeze(0), padding_mask.unsqueeze(0))

print(f"Encoded representation shape: {encoded.shape}")

total_encoder = 0
for name, param in encoder.named_parameters():
    total_encoder += param.numel()
print(f"Encoder parameters: {total_encoder:,}")


Input text: '2018 BMW x7'
Text tokens shape: torch.Size([128])
Padding mask shape: torch.Size([128])
Number of non-padding tokens: 3
Encoded representation shape: torch.Size([1, 128, 128])
Encoder parameters: 588,800


In [ ]:
# Simple Autoregressive decoder for number generation (fixed for log-space)


class TransformerDecoder(nn.Module):

    def __init__(self,
                 vocab_size=16,
                 d_model=128,
                 n_head=4,
                 num_layers=2,
                 max_len=20):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len
        # Create embedding layer and positional encoding
        self.embedding = nn.Embedding(vocab_size, d_model)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model,
                                                   nhead=n_head,
                                                   dim_feedforward=4 * d_model,
                                                   dropout=0.1,
                                                   batch_first=True)
        self.transformer = nn.TransformerDecoder(decoder_layer,
                                                 num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, encoder_output, target_tokens=None):
        if target_tokens is None:
            # Inference.
            return self.generate(encoder_output)
        _, seq_len = target_tokens.shape
        x = self.embedding(target_tokens) * np.sqrt(self.d_model)
        x = x + self.pe[:seq_len, :].unsqueeze(0)
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len,
                                                              device=x.device)
        memory = encoder_output
        x = self.transformer(x, memory, tgt_mask=mask)
        return self.fc_out(x)

    def generate(self, encoder_output, max_tokens=15):
        batch_size = encoder_output.shape[0]
        device = encoder_output.device
        generated = torch.full((batch_size, 1),
                               num_tokenizer.vocab["<BOS>"],
                               dtype=torch.long,
                               device=device)
        for _ in range(max_tokens):
            logits = self.forward(encoder_output, generated)
            # Apply constrained decoding: mask out invalid tokens
            num_prev_tokens = generated.shape[1] - 1  # Exclude <BOS>
            valid_token_ids = num_tokenizer.possible_next_tokens(
                num_prev_tokens)
            # Create mask: set invalid tokens to -inf
            mask = torch.full_like(logits[:, -1, :], float('-inf'))
            mask[:, valid_token_ids] = 0
            logits[:, -1, :] = logits[:, -1, :] + mask
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat((generated, next_token), dim=1)
            if (next_token == num_tokenizer.vocab['<PAD>']).all():
                break
        return generated


decoder = TransformerDecoder(vocab_size=len(num_tokenizer.vocab))

# Generate a number from encoded text
with torch.no_grad():
    generated_tokens = decoder.generate(encoded)
    predicted_price = num_tokenizer.decode(generated_tokens[0])

print(f"Generated tokens: {generated_tokens[0]}")
print(f"Predicted price: ${predicted_price:,.0f}")

Generated tokens: tensor([15, 12,  3, 10,  1,  3,  1, 11, 13,  0,  2, 14])
Predicted price: $0


In [ ]:
class SimpleRegressLm(nn.Module):

    def __init__(self):
        super().__init__()
        self.encoder = TransformerEncoder(vocab_size=len(word_tokenizer.vocab),
                                          d_model=256,
                                          num_layers=2)
        self.decoder = TransformerDecoder(vocab_size=len(num_tokenizer.vocab),
                                          d_model=256,
                                          num_layers=2)

    def forward(self, text_tokens, target_tokens=None):
        encoder_output = self.encoder(text_tokens)
        if target_tokens is None:
            # Inference.
            return self.decoder.generate(encoder_output)
        # Training.
        return self.decoder(encoder_output, target_tokens)

    def predict_batch(self, text_list):
        self.eval()
        with torch.no_grad():
            text_tokens = torch.stack(
                [word_tokenizer.encode(text) for text in text_list])
            device = next(self.parameters()).device
            text_tokens = text_tokens.to(device)
            generated_tokens = self.forward(text_tokens)
            predictions = []
            for tokens in generated_tokens:
                predictions.append(num_tokenizer.decode(tokens))
            return predictions


# Create model
model = SimpleRegressLm()

# Test prediction (before training)
test_description = "2018 BMW x7"
predicted = model.predict_batch([test_description])
print(f"Input: '{test_description}'")
print(f"Predicted price (untrained): ${predicted[0]:,.0f}")

Input: '2018 BMW x7'
Predicted price (untrained): $255,499,999,999,999,990,613,147,648


In [ ]:
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

text_emb_params = model.encoder.embedding.weight.numel()
num_emb_params = model.decoder.embedding.weight.numel()
print(f"Text embedding parameters: {text_emb_params:,}"
      )  # (384,000 = 1500 our vocab size * 256 d_model)
print(f"Number embedding parameters: {num_emb_params:,}"
      )  # (4,096 = 16 our num vocab size * 256 d_model)

total_encoder = 0
total_decoder = 0

for name, param in model.encoder.named_parameters():
    total_encoder += param.numel()
print(f"Encoder parameters: {total_encoder:,}")

for name, param in model.decoder.named_parameters():
    total_decoder += param.numel()
print(f"Decoder parameters: {total_decoder:,}")

print(f"Total parameters: {total_encoder + total_decoder:,}")

Model parameters: 4,079,120
Text embedding parameters: 384,000
Number embedding parameters: 4,096
Encoder parameters: 1,964,032
Decoder parameters: 2,115,088
Total parameters: 4,079,120


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch import optim


class CarPriceDataset(Dataset):

    def __init__(self, df, text_tokenizer, num_tokenizer):
        self.descriptions = df['description'].tolist()
        self.prices = df['price'].tolist()
        self.text_tokenizer = text_tokenizer
        self.num_tokenizer = num_tokenizer

    def __len__(self):
        return len(self.descriptions)

    def __getitem__(self, idx):
        description = self.descriptions[idx]
        price = self.prices[idx]
        description_tokens = self.text_tokenizer.encode(description)
        price_tokens = self.num_tokenizer.encode(price)
        return description_tokens, price_tokens


train_subset = df.head(25000)
dataset = CarPriceDataset(train_subset, word_tokenizer, num_tokenizer)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0003)
criterion = nn.CrossEntropyLoss(ignore_index=num_tokenizer.vocab['<PAD>'])

print(f"Device: {device}")
print(f"Training examples: {len(dataset)}")
print(f"Batches: {len(dataloader)}")

Device: cpu
Training examples: 25000
Batches: 782


In [ ]:
num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    for btach_idx, (description_tokens, price_tokens) in enumerate(dataloader):
        description_tokens = description_tokens.to(device)
        price_tokens = price_tokens.to(device)
        input_tokens = price_tokens[:, :-1]
        target_tokens = price_tokens[:, 1:]
        logits = model(description_tokens, input_tokens)
        loss = criterion(logits.reshape(-1, logits.size(-1)),
                         target_tokens.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

In [ ]:
val_subset = df.iloc[25000:26000]

model.eval()

predictions = []
actuals = []
batch_size = 32

with torch.no_grad():
    for i in range(0, len(val_subset), batch_size):
        batch_data = val_subset.iloc[i:i + batch_size]
        generated = model.predict_batch(batch_data['description'])
        predictions.extend(generated)
        actuals.extend(batch_data['price'])

predictions = np.array(predictions)
actuals = np.array(actuals)
mae = np.mean(np.abs(predictions - actuals))

print(f"Validation MAE: ${mae:,.0f}")

for i in range(5):
    print(f"Description: {val_subset.iloc[i]['description'][:100]}...")
    print(
        f"Actual price: ${actuals[i]:,.0f}, Predicted price: ${predictions[i]:,.0f}"
    )
    print("Difference: ${:,.0f}".format(abs(actuals[i] - predictions[i])))


In [ ]:
import torch.nn.functional as F

# Take a single example from validation set
example_idx = 0
example_description = val_subset.iloc[example_idx]['description']
example_price = val_subset.iloc[example_idx]['price']

print(f"Description: {example_description[:100]}...")
print(f"Price: ${example_price:,.0f}")

text_tokens = word_tokenizer.encode(example_description).unsqueeze(0).to(device)
model.eval()
with torch.no_grad():
    encoder_output = model.encoder(text_tokens)

target_tokens = num_tokenizer.encode(example_price)
print(f"Target price tokens: {target_tokens.tolist()}")
string_tokens = [num_tokenizer.inv_vocab[token_id] for token_id in target_tokens.tolist()]
print(f"Target price string: {''.join(string_tokens)}")

generated = torch.tensor([[num_tokenizer.vocab["<BOS>"]]], device=device)

for step in range(11):
    with torch.no_grad():
        logits = model.decoder(encoder_output, generated)
        next_token_logits = logits[0, -1, :]
        # Apply constrained decoding
        num_prev_tokens = generated.shape[1] - 1
        valid_token_ids = num_tokenizer.possible_next_tokens(num_prev_tokens)
        # Create mask for invalid tokens
        mask = torch.full_like(next_token_logits, float('-inf'))
        mask[valid_token_ids] = 0
        masked_logits = next_token_logits + mask
        # Convert to probabilities
        probs = F.softmax(masked_logits, dim=0)
        # Get the predicted token (greedy decoding)
        next_token_id = masked_logits.argmax().item()
        next_token_char = num_tokenizer.inv_vocab[next_token_id]
        string_tokens_this_far = [num_tokenizer.inv_vocab[token_id] for token_id in generated[0].tolist()[1:]]
        print(f"{step + 1} Generated so far: {''.join(string_tokens_this_far)}")
        print(f"Valid next tokens ({len(valid_token_ids)}): {[num_tokenizer.inv_vocab[tid] for tid in valid_token_ids]}")
        # Show top 5 probabilities for valid tokens
        valid_probs = [(tid, probs[tid].item()) for tid in valid_token_ids]
        valid_probs.sort(key=lambda x: x[1], reverse=True)
        print("Top probabilities:")
        for tid, prob in valid_probs[:5]:
            char = num_tokenizer.inv_vocab[tid]
            bar = "█" * int(prob * 50)
            print(f"  '{char}': {prob:6.2%} {bar}")
        print(f"Predicted: '{next_token_char}'")
        # Add to generated sequence
        generated = torch.cat([generated, torch.tensor([[next_token_id]], device=device)], dim=1)
        # Stop if we hit padding
        if next_token_id == num_tokenizer.vocab['<PAD>']:
            break

final_prediction = num_tokenizer.decode(generated[0].cpu())

print(f"Final predicted price: ${final_prediction:,.0f}")
print(f"Actual price:          ${example_price:,.0f}")
print(f"Error:                 ${abs(final_prediction - example_price):,.0f}")

In [ ]:
# To install 'regress_lm' (or any other Python package) directly from a GitHub repository:
!pip install git+https://github.com/google-deepmind/regress-lm.git


In [ ]:
from regress_lm import rlm

print("Creating official RegressLM model...")

# Determine device (same as our simple model)
official_device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {official_device}")

# Create RegressLM from scratch with similar config to our simple model
# Note: Configuration like max_epochs and optimizer_factory go in from_scratch()
official_model = rlm.RegressLM.from_scratch(device=official_device,
                                            d_model=256,
                                            num_encoder_layers=2,
                                            num_decoder_layers=2,
                                            max_input_len=128,
                                            max_epochs=30,
                                            batch_size=64,
                                            batch_size_per_device=32)

print(
    f"Model parameters: {sum(p.numel() for p in official_model.model.parameters()):,}"
)


In [ ]:
train_examples = [
    core.Example(x=row['description'], y=row['price'])
    for _, row in train_subset.iterrows()
]

validation_examples = [
    core.Example(x=row['description'], y=row['price'])
    for _, row in val_subset.iterrows()
]

official_model.fine_tune(train_examples, validation_examples)

In [ ]:
official_predictions = []
for i, example in enumerate(validation_examples):
    example_input = core.ExampleInput(x=example.x)
    samples = official_model.sample([example_input], num_samples=1)
    pred = float(samples[0][0])
    official_predictions.append(pred)

official_predictions = np.array(official_predictions)
actuals = np.array([e.y for e in validation_examples])

official_mae = np.mean(np.abs(official_predictions - actuals))

print(f"Simple RegressLM MAE:    ${mae:>12,.0f}")
print(f"Official RegressLM MAE:  ${official_mae:>12,.0f}")
print(
    f"Difference: ${abs(mae - official_mae):,.0f} ({((mae - official_mae) / official_mae * 100):+.1f}%)"
)

# Show some example predictions
print(f"Sample predictions:")
for i in range(5):
    print(
        f"{i+1}. Actual: ${actuals[i]:>10,.0f}, Predicted: ${official_predictions[i]:>10,.0f}, Error: ${abs(actuals[i] - official_predictions[i]):>10,.0f}"
    )